# Task 2: Multi-hand recognizer demo (the module Task 3 / Task 5 will import)

Runs `src.recognizer.SignRecognizer` on your webcam. `rec.update(frame, frame_id)` returns
**exactly the agreed Task 2 → Task 3 format** — `{"frame_id": ..., "hands": [{"points", "confidence", "handedness", "box"}, ...]}`.
The agreed format has no field for the recognised word, so it is available separately as `rec.signs`
(one entry per hand, same order).

What you see: a box + the 21 smoothed points per hand, the temporary track id, handedness, confidence,
the current guess, and a green box + printed line when a sign is **confirmed**. Try two hands at once.

Press **q** in the video window to quit. Kernel: **Python (ipcv-task2)**.

**Debugging a word that gets mistaken for another:** in the first code cell set `TRUE_WORD` (e.g. `"Yesterday"`) and
`HAND_USED` (`"left"` / `"right"`), run, sign that word ~10 times, press **q**. Under each hand you will see the voted
hand, whether its features were mirrored, and the model's top-3 guesses with probabilities. A log is saved to
`data/live_logs/` so the exact reasoning can be inspected afterwards.

In [ ]:
import sys, os, json, time
sys.path.append(os.path.abspath(".."))

import cv2
from src.recognizer import SignRecognizer

# --- optional debugging: label what you are about to sign so the log is easy to read ---
TRUE_WORD = None   # e.g. "Yesterday"  (the word you are signing in this run)
HAND_USED = None   # "left" or "right"

rec = SignRecognizer()   # loads models/hand_landmarker.task and models/sign_classifier.joblib
print("Recognizer ready. Classes:", rec.classes)

In [ ]:
FONT = cv2.FONT_HERSHEY_SIMPLEX
cap = cv2.VideoCapture(0)
frame_id, shown_example = 0, False
ms_hist = []
log_rows = []   # what the model thought on every classified frame (saved at the end)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame grab failed, stopping.")
            break

        out = rec.update(frame, frame_id)      # <- the agreed Task 2 -> Task 3 dict (coordinates refer to the RAW frame)
        frame_id += 1
        ms_hist = (ms_hist + [rec.timing["total_ms"]])[-60:]

        if out["hands"] and not shown_example:
            print("Example output for one frame (truncated):")
            print(json.dumps(out)[:700], "...")
            shown_example = True

        view = cv2.flip(frame, 1)              # mirror for display only, so mirror x when drawing
        h, w = view.shape[:2]
        for hand, sign in zip(out["hands"], rec.signs):
            for (x, y, _z) in hand["points"]:
                cv2.circle(view, (int((1.0 - x) * w), int(y * h)), 3, (0, 255, 0), -1)
            bx, by, bw, bh = hand["box"]
            x0 = 1.0 - (bx + bw)
            color = (0, 255, 0) if sign["confirmed"] else (0, 200, 255)
            cv2.rectangle(view, (int(x0 * w), int(by * h)), (int((x0 + bw) * w), int((by + bh) * h)), color, 2)
            tag = f"#{sign['track_id']} {hand['handedness']} {hand['confidence']:.2f}"
            cv2.putText(view, tag, (int(x0 * w), max(15, int(by * h) - 28)), FONT, 0.55, color, 2)
            if sign["guess"]:
                cv2.putText(view, f"{sign['guess']} {sign['guess_confidence']:.2f}",
                            (int(x0 * w), max(15, int(by * h) - 8)), FONT, 0.6, color, 2)
            if sign["confirmed"]:
                print(f"[{time.strftime('%H:%M:%S')}] CONFIRMED hand #{sign['track_id']} ({hand['handedness']}): {sign['label']}")
            if sign.get("top3"):
                dbg = f"{sign['hand']}{' (mirrored)' if sign['mirrored'] else ''}: " + ", ".join(f"{l} {p:.2f}" for l, p in sign["top3"])
                cv2.putText(view, dbg, (int(x0 * w), min(h - 8, int((by + bh) * h) + 18)), FONT, 0.5, color, 1)
                log_rows.append({"t": round(time.time(), 3), "true_word": TRUE_WORD, "hand_used": HAND_USED,
                                 "track": sign["track_id"], "voted_hand": sign["hand"], "mirrored": sign["mirrored"],
                                 "guess": sign["guess"], "conf": sign["guess_confidence"],
                                 "top3": json.dumps(sign["top3"]), "confirmed": sign["confirmed"]})

        avg = sum(ms_hist) / len(ms_hist)
        cv2.putText(view, f"{len(out['hands'])} hand(s) | {avg:.0f} ms/frame (~{1000 / max(avg, 1e-3):.0f} fps)",
                    (10, 25), FONT, 0.6, (255, 255, 255), 2)
        cv2.imshow("Task 2 - multi-hand recognizer (q to quit)", view)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.waitKey(1)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

if log_rows:
    import pandas as pd
    os.makedirs("../data/live_logs", exist_ok=True)
    path = f"../data/live_logs/{TRUE_WORD or 'unlabelled'}_{HAND_USED or 'any'}_{time.strftime('%H%M%S')}.csv"
    pd.DataFrame(log_rows).to_csv(path, index=False)
    print(f"Saved {len(log_rows)} rows of model thinking to {path}")

In [ ]:
rec.close()
print("Recognizer closed.")